# PariShiksha Study Assistant v2.0
**Week 10 · Dense Embeddings + Chroma + Strict Grounding · IIT Gandhinagar**

Run cells in order. Cell 6 embeds into Chroma on first run and skips on restart.

In [ ]:
import subprocess, sys

pkgs = [
    'langchain==0.3.*', 'langchain-community==0.3.*',
    'langchain-openai==0.2.*', 'langchain-anthropic==0.2.*',
    'chromadb==0.5.*', 'openai>=1.50', 'anthropic>=0.39',
    'tiktoken', 'rank_bm25', 'sentence-transformers',
    'cohere', 'python-dotenv', 'pandas', 'numpy',
    'pymupdf', 'pdfplumber',
]
for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '-q'])
print('Dependencies ready.')

In [ ]:
import os, re, json, warnings, time
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict
from collections import Counter

import tiktoken
import fitz
import pdfplumber
from rank_bm25 import BM25Okapi

import chromadb
from chromadb.config import Settings

from langchain.schema import Document
from langchain.prompts import PromptTemplate
from langchain_anthropic import ChatAnthropic
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()

warnings.filterwarnings('ignore')

DATA_DIR   = Path('data');   DATA_DIR.mkdir(exist_ok=True)
CHROMA_DIR = Path('chroma_wk10'); CHROMA_DIR.mkdir(exist_ok=True)

PDF_PATHS = {
    'ch08_motion': DATA_DIR / 'iesc108.pdf',
    'ch09_force':  DATA_DIR / 'iesc109.pdf',
}

TOKENIZER = tiktoken.get_encoding('cl100k_base')
def count_tokens(text: str) -> int:
    return len(TOKENIZER.encode(text))

print('Imports OK.')
for name, path in PDF_PATHS.items():
    print(f'  {name}: {"found" if path.exists() else "MISSING — place PDF in data/"}')

---
## Stage 1 — Token-Aware Chunking
tiktoken cl100k\_base · 250 tokens · 40 overlap · content\_type: prose / worked\_example / question\_or\_exercise

In [ ]:
def extract_pages(pdf_path: Path, chapter: str) -> List[Dict]:
    pages = []
    doc = fitz.open(str(pdf_path))
    with pdfplumber.open(str(pdf_path)) as plumb:
        for i, page in enumerate(doc):
            text = page.get_text('text').strip()
            if len(text) < 100:
                text = (plumb.pages[i].extract_text() or '').strip()
            pages.append({'chapter': chapter, 'page': i + 1, 'text': text})
    doc.close()
    return pages

raw_corpus: List[Dict] = []
for chapter_key, pdf_path in PDF_PATHS.items():
    if not pdf_path.exists():
        print(f'[SKIP] {pdf_path} — place PDF in data/')
        continue
    pages = extract_pages(pdf_path, chapter_key)
    raw_corpus.extend(pages)
    print(f'Extracted {len(pages)} pages from {chapter_key}')
print(f'Total pages: {len(raw_corpus)}')

In [ ]:
CHUNK_TOKENS   = 250
OVERLAP_TOKENS = 40

_EX = re.compile(r'\b(example|solution|solved\s+example)\b', re.IGNORECASE)
_QS = re.compile(r'^\s*(Q\.?\s*\d+|\d+\.\s+[A-Z]|Exercise)', re.MULTILINE | re.IGNORECASE)
_HD = re.compile(r'^(\d+\.\d*\s+)?[A-Z][A-Z\s]{4,}$')

def classify(text: str) -> str:
    t = text.strip()
    if _EX.search(t):                    return 'worked_example'
    if _QS.search(t):                    return 'question_or_exercise'
    if _HD.match(t) and len(t) < 80:     return 'heading'
    return 'prose'

def make_chunks(pages: List[Dict]) -> List[Dict]:
    chunks, cid = [], 0
    pending_heading = ''
    buf_words, buf_meta = [], {}
    current_section = 'preamble'

    def flush():
        nonlocal cid, buf_words, pending_heading
        if not buf_words: return
        body = ' '.join(buf_words)
        full = (pending_heading + ' ' + body).strip() if pending_heading else body
        pending_heading = ''
        chunks.append({'chunk_id': f'chunk_{cid:04d}', **buf_meta,
                       'token_count': count_tokens(full), 'text': full})
        cid += 1
        buf_words[:] = buf_words[-OVERLAP_TOKENS:]

    for page in pages:
        for raw in re.split(r'\n{2,}', page['text']):
            block = raw.strip()
            if len(block) < 20: continue
            ctype = classify(block)

            if ctype == 'heading':
                flush()
                current_section = block
                pending_heading = block
                buf_meta = {'chapter': page['chapter'], 'page': page['page'],
                            'section': current_section, 'content_type': 'prose'}
                continue

            if ctype == 'worked_example':
                flush()
                full = (pending_heading + ' ' + block).strip() if pending_heading else block
                pending_heading = ''
                chunks.append({'chunk_id': f'chunk_{cid:04d}',
                               'chapter': page['chapter'], 'page': page['page'],
                               'section': current_section,
                               'content_type': 'worked_example',
                               'token_count': count_tokens(full), 'text': full[:2000]})
                cid += 1
                continue

            buf_meta = {'chapter': page['chapter'], 'page': page['page'],
                        'section': current_section, 'content_type': ctype}
            buf_words.extend(block.split())
            if count_tokens(' '.join(buf_words)) >= CHUNK_TOKENS:
                flush()

    flush()
    return chunks

wk10_chunks = make_chunks(raw_corpus)

with open('wk10_chunks.json', 'w') as f:
    json.dump(wk10_chunks, f, indent=2)

toks = [c['token_count'] for c in wk10_chunks]
print(f'Chunks: {len(wk10_chunks)}')
print(f'Content types: {dict(Counter(c["content_type"] for c in wk10_chunks))}')
print(f'Tokens — min:{min(toks)} max:{max(toks)} mean:{np.mean(toks):.0f}')
print('Saved: wk10_chunks.json')

In [ ]:
def word_tok(text): return text.lower().split()

bm25 = BM25Okapi([word_tok(c['text']) for c in wk10_chunks])

def bm25_get(query, k=3):
    scores = bm25.get_scores(word_tok(query))
    idx = np.argsort(scores)[::-1][:k]
    return [dict(**wk10_chunks[i], bm25_score=round(float(scores[i]),4)) for i in idx]

spot = [
    'What is the SI unit of acceleration?',
    "Newton's first law of motion",
    'Define inertia',
    'Area under velocity-time graph',
    'Newton second law momentum',
]
rlog = []
for q in spot:
    res = bm25_get(q)
    print(f'Q: {q}')
    for r in res:
        print(f'  [{r["chunk_id"]}] {r["bm25_score"]} {r["content_type"]} | {r["text"][:70]}...')
    rlog.append({'query': q, 'top3': [{'chunk_id': r['chunk_id'],
        'score': r['bm25_score'], 'preview': r['text'][:100]} for r in res]})

with open('retrieval_log_bm25.json', 'w') as f:
    json.dump(rlog, f, indent=2)
print('BM25 sanity done.')

---
## Stage 2 — Embed and Retrieve
text-embedding-3-small → Chroma PersistentClient · cosine · skips re-embedding on restart

In [ ]:
OPENAI_KEY = os.environ.get('OPENAI_API_KEY', '')
if not OPENAI_KEY:
    raise EnvironmentError('Set OPENAI_API_KEY in .env')

oai = OpenAI(api_key=OPENAI_KEY)
EMBED_MODEL = 'text-embedding-3-small'   # locked — same at index and query time

chroma = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)
col = chroma.get_or_create_collection('parishiksha_wk10',
                                       metadata={'hnsw:space': 'cosine'})

if col.count() == 0:
    print(f'Embedding {len(wk10_chunks)} chunks...')
    BATCH = 100
    for s in range(0, len(wk10_chunks), BATCH):
        batch = wk10_chunks[s:s+BATCH]
        resp  = oai.embeddings.create(model=EMBED_MODEL, input=[c['text'] for c in batch])
        col.add(
            ids        = [c['chunk_id'] for c in batch],
            documents  = [c['text'] for c in batch],
            embeddings = [r.embedding for r in resp.data],
            metadatas  = [{'chapter': c['chapter'], 'section': c['section'],
                           'page': str(c['page']), 'content_type': c['content_type']} for c in batch],
        )
        print(f'  {s+len(batch)}/{len(wk10_chunks)}')
    print('Embedding complete and persisted.')
else:
    print(f'Collection already has {col.count()} chunks — skipping re-embed.')

In [ ]:
def dense_get(query: str, k: int = 5) -> List[Dict]:
    q_emb = oai.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
    res = col.query(query_embeddings=[q_emb], n_results=k,
                    include=['documents','metadatas','distances'])
    out = []
    for i, cid in enumerate(res['ids'][0]):
        out.append({'chunk_id': cid,
                    'text': res['documents'][0][i],
                    'metadata': res['metadatas'][0][i],
                    'similarity': round(1 - res['distances'][0][i], 4)})
    return out

ten = [
    ('What is the SI unit of acceleration?',               True),
    ("State Newton's first law of motion",                 True),
    ('What is the formula for acceleration?',              True),
    ('Define momentum and give its SI unit',               True),
    ('Straight line on distance-time graph indicates?',    True),
    ('Area under velocity-time graph represents?',         True),
    ("State Newton's third law",                           True),
    ('Difference between speed and velocity?',             True),
    ('Define inertia',                                     False),  # known Wk9 failure
    ('How does Newton second law connect to momentum?',    False),  # multi-hop
]

dense_log = []
for query, expected_hit in ten:
    res = dense_get(query)
    t1 = res[0]
    print(f'[{"YES" if expected_hit else "NO "}] {query}')
    print(f'       top-1: [{t1["chunk_id"]}] sim={t1["similarity"]} | {t1["text"][:65]}...')
    dense_log.append({'query': query, 'top1_chunk_id': t1['chunk_id'],
                      'top1_sim': t1['similarity'], 'preview': t1['text'][:120],
                      'contains_answer': expected_hit})

with open('retrieval_log.json', 'w') as f:
    json.dump(dense_log, f, indent=2)
print('Saved: retrieval_log.json')

---
## Stage 3 — Strict Grounding + ask()
Permissive prompt shown first so hallucination is visible. Strict prompt follows.

In [ ]:
ANTH_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
if not ANTH_KEY:
    raise EnvironmentError('Set ANTHROPIC_API_KEY in .env')

llm = ChatAnthropic(model='claude-haiku-4-5', temperature=0,
                    api_key=ANTH_KEY, max_tokens=512)

def build_ctx(chunks):
    return '\n\n---\n\n'.join(f'[{c["chunk_id"]}]\n{c["text"]}' for c in chunks)

PERM = 'Answer using the context below.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:'

demo_qs = [
    'What is the formula for acceleration?',
    'Explain how photosynthesis works.',     # OOS — expected hallucination
    'Define inertia.',
]

print('=== PERMISSIVE PROMPT (v1) ===')
for q in demo_qs:
    chunks = dense_get(q)
    resp = llm.invoke(PERM.format(context=build_ctx(chunks), question=q))
    print(f'\nQ: {q}')
    print(f'A: {resp.content[:200]}')

In [ ]:
REFUSAL = "I don't have that in my study materials."

STRICT = (
    'You are a study assistant for Class 9 NCERT Science.\n'
    '\n'
    'RULES:\n'
    '1. Answer ONLY using information in the retrieved chunks below.\n'
    '2. If the answer is not in the chunks, reply with exactly: '
    '"I don\'t have that in my study materials."\n'
    '3. After every factual claim cite the chunk: [Source: chunk_id].\n'
    '4. Do NOT use numbers, values, or facts from the question itself '
    'unless they also appear in a retrieved chunk.\n'
    '5. Keep the answer under 120 words.\n'
    '\n'
    'RETRIEVED CHUNKS:\n'
    '{context}\n'
    '\n'
    'STUDENT QUESTION:\n'
    '{question}\n'
    '\n'
    'ANSWER:'
)

SIM_THRESHOLD = 0.35

def ask(question: str, k: int = 5) -> Dict:
    retrieved = dense_get(question, k)
    max_sim = max(r['similarity'] for r in retrieved) if retrieved else 0.0
    if max_sim < SIM_THRESHOLD:
        return {'answer': REFUSAL, 'sources': [], 'chunk_ids': [],
                'cited_ids': [], 'max_sim': max_sim, 'threshold_hit': True}
    ctx = build_ctx(retrieved)
    resp = llm.invoke(STRICT.format(context=ctx, question=question))
    answer = resp.content.strip()
    cited = re.findall(r'\[Source:\s*(chunk_\d+)\]', answer)
    return {'answer': answer, 'sources': [r['metadata'] for r in retrieved[:3]],
            'chunk_ids': [r['chunk_id'] for r in retrieved],
            'cited_ids': cited, 'max_sim': max_sim, 'threshold_hit': False}

print('=== STRICT PROMPT (v2) ===')
for q in demo_qs:
    r = ask(q)
    print(f'\nQ: {q}')
    print(f'A: {r["answer"][:220]}')
    print(f'Cited: {r["cited_ids"]}  |  max_sim: {r["max_sim"]}')

---
## Stage 4 — 12-Question Evaluation
6 direct · 3 paraphrased · 3 out-of-scope (including 1 plausibly-answerable) · scored on 3 axes

In [ ]:
eval_set = [
    # Direct factual
    {'id':'E01','q':'What is the SI unit of acceleration?','type':'direct','expected':'m/s²'},
    {'id':'E02','q':"State Newton's first law of motion.", 'type':'direct',
     'expected':'Every object remains at rest or uniform motion unless acted on by external force'},
    {'id':'E03','q':'What is the formula for calculating acceleration?','type':'direct',
     'expected':'a = (v - u) / t'},
    {'id':'E04','q':'Define momentum and give its SI unit.','type':'direct',
     'expected':'momentum = mass x velocity; kg m/s'},
    {'id':'E05','q':'What does a straight line on a distance-time graph indicate?',
     'type':'direct','expected':'uniform motion constant speed'},
    {'id':'E06','q':'Define inertia.','type':'direct',
     'expected':'tendency to resist changes in state of rest or motion'},
    # Paraphrased
    {'id':'E07','q':'If nothing pushes or pulls a moving object, what happens to it?',
     'type':'paraphrased','expected':'continues in straight line at same speed'},
    {'id':'E08','q':'A steeper slope on a distance-time graph means the object is doing what?',
     'type':'paraphrased','expected':'moving faster greater speed'},
    {'id':'E09','q':'Two objects collide with no outside forces — what quantity stays the same?',
     'type':'paraphrased','expected':'total momentum conserved'},
    # Out of scope
    {'id':'E10','q':'Explain how photosynthesis works in plants.','type':'out_of_scope',
     'expected':REFUSAL},
    {'id':'E11','q':'What is quantum entanglement?','type':'out_of_scope','expected':REFUSAL},
    # Plausibly-answerable OOS: formula in corpus, Moon g value is not
    {'id':'E12','q':'Using F = ma, calculate the weight of a 70 kg astronaut on the Moon where g = 1.62 m/s².',
     'type':'out_of_scope','expected':REFUSAL},
]

print(f'{len(eval_set)} questions loaded.')
for q in eval_set:
    print(f'  {q["id"]} [{q["type"]}] {q["q"][:55]}')

In [ ]:
import csv

print('Running evaluation...')
raw_rows = []

for item in eval_set:
    print(f'  {item["id"]}...', end=' ')
    result = ask(item['q'])
    raw_rows.append({
        'id': item['id'], 'question': item['q'], 'question_type': item['type'],
        'expected': item['expected'], 'generated': result['answer'],
        'chunk_ids': ', '.join(result['chunk_ids'][:3]),
        'cited_ids': ', '.join(result['cited_ids']),
        'max_sim': result['max_sim'],
    })
    print('done')
    time.sleep(0.5)

fields = ['id','question','question_type','expected','generated','chunk_ids','cited_ids','max_sim']
with open('eval_raw.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(raw_rows)
print('Saved: eval_raw.csv')

In [ ]:
scored = []

for row in raw_rows:
    refused = REFUSAL in row['generated']
    if row['question_type'] == 'out_of_scope':
        correctness, grounded, refused_oos = \
            ('yes' if refused else 'no'), (1 if refused else 0), ('yes' if refused else 'no')
    else:
        refused_oos = 'N/A'
        ek = set(row['expected'].lower().split())
        gk = set(row['generated'].lower().split())
        ov = len(ek & gk) / max(len(ek), 1)
        correctness = 'yes' if ov > 0.4 else ('partial' if ov > 0.15 else 'no')
        grounded = 1 if row['cited_ids'] else 0

    scored.append({'id': row['id'], 'question': row['question'],
                   'question_type': row['question_type'],
                   'generated': row['generated'][:200], 'expected': row['expected'],
                   'chunk_ids': row['chunk_ids'],
                   'correctness': correctness, 'grounded': grounded,
                   'refused_when_oos': refused_oos, 'notes': ''})

fields2 = ['id','question','question_type','generated','expected','chunk_ids',
           'correctness','grounded','refused_when_oos','notes']
with open('eval_scored.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields2); w.writeheader(); w.writerows(scored)

df = pd.DataFrame(scored)
total = len(df)
correct  = (df.correctness == 'yes').sum()
grounded_n = (df.grounded == 1).sum()
oos = df[df.question_type == 'out_of_scope']
refused = (oos.refused_when_oos == 'yes').sum()

print('=== STAGE 4 RESULTS ===')
print(f'Correct:       {correct}/{total}')
print(f'Grounded:      {grounded_n}/{total}')
print(f'Refused OOS:   {refused}/{len(oos)}')
print('Saved: eval_scored.csv')

---
## Stage 5 — One Targeted Fix with Honest Delta
Fix: similarity threshold guard already wired into ask(). Re-run full 12-Q set.

In [ ]:
# ask() already has the threshold guard (SIM_THRESHOLD = 0.35)
# and the no-calc-from-query instruction in STRICT.
# Re-run is identical call — threshold catches E12.

print('Re-running evaluation with fix applied...')
v2_rows = []

for item in eval_set:
    print(f'  {item["id"]}...', end=' ')
    result = ask(item['q'])
    refused = REFUSAL in result['answer']
    if item['type'] == 'out_of_scope':
        correctness, grounded, refused_oos = \
            ('yes' if refused else 'no'), (1 if refused else 0), ('yes' if refused else 'no')
    else:
        refused_oos = 'N/A'
        ek = set(item['expected'].lower().split())
        gk = set(result['answer'].lower().split())
        ov = len(ek & gk) / max(len(ek), 1)
        correctness = 'yes' if ov > 0.4 else ('partial' if ov > 0.15 else 'no')
        grounded = 1 if result['cited_ids'] else 0
    v2_rows.append({'id': item['id'], 'question': item['q'],
                    'question_type': item['type'],
                    'generated': result['answer'][:200], 'expected': item['expected'],
                    'chunk_ids': ', '.join(result['chunk_ids'][:3]),
                    'correctness': correctness, 'grounded': grounded,
                    'refused_when_oos': refused_oos,
                    'threshold_hit': result['threshold_hit'],
                    'max_sim': result['max_sim']})
    print('done')
    time.sleep(0.5)

fields3 = ['id','question','question_type','generated','expected','chunk_ids',
           'correctness','grounded','refused_when_oos','threshold_hit','max_sim']
with open('eval_v2_scored.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields3); w.writeheader(); w.writerows(v2_rows)

df2 = pd.DataFrame(v2_rows)
c2 = (df2.correctness=='yes').sum()
g2 = (df2.grounded==1).sum()
oos2 = df2[df2.question_type=='out_of_scope']
r2 = (oos2.refused_when_oos=='yes').sum()

print('\n=== DELTA: v1 → v2 ===')
print(f'Correct:     {correct}/{total}  →  {c2}/{total}')
print(f'Grounded:    {grounded_n}/{total}  →  {g2}/{total}')
print(f'Refused OOS: {refused}/{len(oos)}  →  {r2}/{len(oos2)}')
print('Saved: eval_v2_scored.csv')

---
## Done
All stage files generated. Add Loom link to README.md and submit.